# Phase 0: Exploring MovieLens

Goal: get a feel for the data before modelling anything. By the end you should be able to answer:

1. How sparse is the user × movie matrix?
2. What does the rating distribution look like? Do people use the whole 0.5–5 scale?
3. How long is the tail? What share of ratings goes to the most popular movies?
4. How active are users? How many ratings does a typical user have?
5. Can you trust a movie's average rating when it has only 2 ratings?
6. How are ratings spread over time, and why does that matter for train/test splits?

Run `python training/download_data.py` from `apps/recommender` first.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Notebook runs from training/notebooks; data lives in apps/recommender/data.
DATA = Path.cwd().resolve().parents[1] / "data" / "raw" / "ml-latest-small"
assert DATA.exists(), f"{DATA} not found; run training/download_data.py first"

ratings = pd.read_csv(DATA / "ratings.csv")
movies = pd.read_csv(DATA / "movies.csv")
tags = pd.read_csv(DATA / "tags.csv")
links = pd.read_csv(DATA / "links.csv")

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit="s")

for name, df in [("ratings", ratings), ("movies", movies), ("tags", tags), ("links", links)]:
    print(f"{name:8s} {df.shape}")

In [ ]:
ratings.head()

In [ ]:
movies.head()

## 1. Sparsity

If every user had rated every movie, the matrix would be full. Collaborative filtering exists because it is mostly empty.

In [ ]:
n_users = ratings["userId"].nunique()
n_rated_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)
density = n_ratings / (n_users * n_rated_movies)

print(f"users:          {n_users:,}")
print(f"movies (rated): {n_rated_movies:,} of {len(movies):,} in catalog")
print(f"ratings:        {n_ratings:,}")
print(f"density:        {density:.2%}  (sparsity {1 - density:.2%})")

## 2. Rating distribution

In [ ]:
counts = ratings["rating"].value_counts().sort_index()
ax = counts.plot.bar(figsize=(8, 3), title="Rating distribution")
ax.set_xlabel("rating")
ax.set_ylabel("count")
plt.show()

print(f"mean rating: {ratings['rating'].mean():.3f}")
print(f"share of whole-star ratings: {(ratings['rating'] % 1 == 0).mean():.1%}")

## 3. The long tail (item popularity)

In [ ]:
per_movie = ratings.groupby("movieId").size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(np.arange(len(per_movie)), per_movie.values)
axes[0].set(title="Ratings per movie (sorted)", xlabel="movie rank", ylabel="# ratings", yscale="log")

cum_share = per_movie.cumsum() / per_movie.sum()
axes[1].plot(np.arange(len(cum_share)) / len(cum_share), cum_share.values)
axes[1].set(title="Cumulative share of ratings", xlabel="fraction of movies (most popular first)", ylabel="fraction of ratings")
plt.show()

top10 = cum_share.iloc[int(len(cum_share) * 0.10) - 1]
print(f"top 10% of movies receive {top10:.1%} of all ratings")
print(f"movies with exactly 1 rating: {(per_movie == 1).sum():,} ({(per_movie == 1).mean():.1%})")
print(f"median ratings per movie: {per_movie.median():.0f}")

In [ ]:
# The head of the distribution: what a pure popularity recommender would show everyone.
per_movie.head(15).rename("n_ratings").to_frame().join(movies.set_index("movieId")["title"])

## 4. User activity

In [ ]:
per_user = ratings.groupby("userId").size()
ax = per_user.plot.hist(bins=50, log=True, figsize=(8, 3), title="Ratings per user")
ax.set_xlabel("# ratings")
plt.show()

print(per_user.describe().round(1))

In [ ]:
# Do users have different personal scales? This is what the 'user bias' term captures.
user_mean = ratings.groupby("userId")["rating"].mean()
ax = user_mean.plot.hist(bins=40, figsize=(8, 3), title="Mean rating per user")
ax.set_xlabel("mean rating")
plt.show()

## 5. Can you trust a movie's average?

Movies with very few ratings have extreme averages. Sorting by raw mean puts obscure 5.0 movies on top. Phase 1's bias model fixes this with regularization (shrinking toward the global mean).

In [ ]:
movie_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"]).join(movies.set_index("movieId")["title"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(movie_stats["count"], movie_stats["mean"], s=4, alpha=0.3)
ax.set(xscale="log", xlabel="# ratings (log)", ylabel="mean rating", title="Mean rating vs popularity")
plt.show()

print("Top by raw mean:")
display(movie_stats.sort_values("mean", ascending=False).head(5))
print("Top by raw mean, with at least 50 ratings:")
display(movie_stats[movie_stats["count"] >= 50].sort_values("mean", ascending=False).head(5))

## 6. Ratings over time

A random train/test split lets the model see the future: it trains on ratings made after the ones it is tested on. A time-based split is closer to how the system is used: learn from the past, predict what comes next.

In [ ]:
by_year = ratings.groupby(ratings["timestamp"].dt.year).size()
ax = by_year.plot.bar(figsize=(10, 3), title="Ratings per year")
ax.set_xlabel("year")
plt.show()

print(f"range: {ratings['timestamp'].min():%Y-%m-%d} → {ratings['timestamp'].max():%Y-%m-%d}")
q80 = ratings["timestamp"].quantile(0.8)
print(f"80% of ratings happen before {q80:%Y-%m-%d} (a candidate train/test cutoff)")

# How many test-period users would be unseen in training? That is the cold-start problem.
train_users = set(ratings.loc[ratings["timestamp"] < q80, "userId"])
test_users = set(ratings.loc[ratings["timestamp"] >= q80, "userId"])
print(f"test users never seen in train: {len(test_users - train_users)} of {len(test_users)}")

## 7. Content features: genres and tags

These feed the content-based recommender in phase 2 and the movie tower in phase 4.

In [ ]:
genres = movies["genres"].str.split("|").explode()
ax = genres.value_counts().plot.barh(figsize=(8, 5), title="Movies per genre")
ax.invert_yaxis()
plt.show()

movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$").astype("float")
print(f"movies with a parseable year: {movies['year'].notna().mean():.1%}")
print(f"movies with at least one tag: {movies['movieId'].isin(tags['movieId']).mean():.1%}")
print(f"distinct tags: {tags['tag'].str.lower().nunique():,}")
tags["tag"].str.lower().value_counts().head(15)

## Takeaways

Write your own answers to the six questions at the top here before moving on to phase 1.